# 03 · Evidence Validation & Recommendation Ranking

Rank only candidates with cited source support. Technical replacement parts require an exact known model and explicit compatibility evidence. Scores are ranking heuristics—not likelihoods of safety or correctness.

In [ ]:
%run 02_rag_vector_search.ipynb

In [ ]:
def citation_exists(project_id,chunk_id):
    with db_connect() as cx:
        return cx.execute('''SELECT 1 FROM chunks c JOIN documents d ON c.document_id=d.id
            WHERE c.id=? AND c.project_id=? AND d.status='ready' ''',(chunk_id,ensure_project_id(project_id))).fetchone() is not None

def rank_candidates(payload):
    req=RecommendationInput.model_validate(payload)
    pid=str(req.project_id)
    warnings=[];accepted=[];rejected=[]
    for candidate in req.candidates:
        # Deterministic hard constraint. Never order a part without confirmed exact model compatibility.
        if candidate.kind=='replacement_part':
            if not req.device_model:
                rejected.append({'candidate_id':candidate.candidate_id,'reason':'Missing exact equipment model: diagnose and clarify first'})
                continue
            if req.device_model.casefold() not in [s.casefold() for s in candidate.compatible_models]:
                rejected.append({'candidate_id':candidate.candidate_id,'reason':'Candidate does not explicitly support the requested model'})
                continue
        question=req.query+' '+candidate.name+' '+candidate.description
        hits=hybrid_search(pid,question,req.domain,top_k=20)
        valid=[]
        identity=[candidate.name]+candidate.identifying_terms
        for hit in hits:
            content=hit['excerpt'].casefold()
            if not any(term and term.casefold() in content for term in identity):continue
            if not all(t.casefold() in content for t in candidate.required_terms):continue
            if candidate.kind=='replacement_part':
                if req.device_model.casefold() not in content:continue
                # Both the candidate identifier and compatible model must appear in same evidence chunk.
            if not citation_exists(pid,hit['chunk_id']):continue
            valid.append(hit)
        if not valid:
            rejected.append({'candidate_id':candidate.candidate_id,'reason':'No supporting source passage for candidate and constraints'})
            continue
        # Token-based query alignment + evidence RRF. Not a verified compatibility probability.
        overlap=len(set(tokens(req.query)) & set(tokens(candidate.name+' '+candidate.description)))
        query_norm=max(len(set(tokens(req.query))),1)
        relevance=min(overlap/query_norm,1)
        evidence_quality=min(valid[0]['score'],1)
        rank_score=round(0.5*evidence_quality+0.5*relevance,4)
        row={'candidate_id':candidate.candidate_id,'name':candidate.name,
             'rank_score':rank_score,'status':'evidence_supported',
             'evidence':valid[:3],
             'why':'Candidate matched the provided constraints and named supporting source passages.',
             'caution':'Review the cited specifications and diagnostics; this is not a verified fault diagnosis.'}
        accepted.append(row)
    accepted.sort(key=lambda x:(-x['rank_score'],x['candidate_id']))
    if rejected:warnings.append(f'{len(rejected)} candidates were withheld because required evidence or compatibility was missing')
    if not accepted:warnings.append('No supported candidate. Ask for more diagnostics or authoritative specifications before recommending procurement.')
    with db_connect() as cx:
        for r in accepted[:req.top_k]:
            cx.execute('INSERT INTO recommendations VALUES (?,?,?,?,?,?,?,?)',
                       (str(uuid.uuid4()),pid,req.domain,req.query,r['candidate_id'],r['rank_score'],
                        json.dumps([e['chunk_id'] for e in r['evidence']]),NOW()))
    return {'recommendations':accepted[:req.top_k], 'withheld':rejected, 'warnings':warnings,
            'scoring_note':'Heuristic ranking relevance; not a probability or independent compatibility certification.'}

def evidence_from_hits(project_id,hits):
    saved=[]; pid=ensure_project_id(project_id)
    with db_connect() as cx:
        for hit in hits:
            if not citation_exists(pid,hit['chunk_id']):continue
            eid=str(uuid.uuid4())
            cx.execute('INSERT INTO evidence VALUES (?,?,?,?,?,?,?,?,?)',
                       (eid,pid,hit['domain'],hit['chunk_id'],'document',hit['excerpt'][:500],
                        hit['source_url'],json.dumps({'citation':hit['citation']}),NOW()))
            saved.append(eid)
    return saved

print('Evidence-safe recommender ready')